# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! pip install seaborn
! pip install opencc
! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

# data_path = './data'
data_path = './drive/MyDrive/NLP_HW2'

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [ ]:
import random
import re

# check true answer
def true_answer(expr: str) -> str:
    return str(eval(expr[:-1]))

# generate wrong answer
def make_wrong_answer(expr: str) -> str:
    ans = int(true_answer(expr))
    # bias: [-9..-1, +1..+9], avoid 0
    delta = random.choice([d for d in range(-9,10) if d != 0])
    wrong = ans + delta
    return str(wrong)

# 20% wrong data
rng = np.random.default_rng(42)
mask = rng.random(len(df_train)) < 0.20

df_train_noisy = df_train.copy()
df_train_noisy["tgt"] = df_train_noisy["src"].apply(true_answer)

df_train_noisy.loc[mask, "tgt"] = df_train_noisy.loc[mask, "src"].apply(make_wrong_answer)

print("Total training data：", len(df_train_noisy))
print("Wrong ratio：", (df_train_noisy["tgt"] != df_train_noisy["src"].apply(true_answer)).mean())
df_train_noisy.to_csv(f"{data_path}/arithmetic_train_20pct_noisy.csv", index=False)

df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train_20pct_noisy.csv'))


Total training data： 2369250
Wrong ratio： 0.19987379972565159


In [ ]:
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1181


In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite
chars = sorted(set("".join(df_train['src'].tolist())))
tokens = ['<pad>', '<eos>'] + chars # '<pad>': 0,'<eos>': 1
char_to_id = {ch:i for i, ch in enumerate(tokens)}
id_to_char = {i:ch for ch, i in char_to_id.items()}

vocab_size = len(char_to_id)
print('Vocab size {}'.format(vocab_size))

Vocab size 18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:


# Write your code here
# 產生 x/y 的 id 清單。序列為：src + <eos>，x=[:-1]，y=[1:]
def encode_with_eos(s):
    ids = [char_to_id[c] for c in s] + [char_to_id['<eos>']]
    x_ids = ids[:-1]
    y_ids = ids[1:]
    # 只在 '=' 之後計算 loss：把 '=' 之前的 y 標成 <pad>
    eq_pos = s.index('=')  # 一定存在
    for t in range(eq_pos):  # y 的前 eq_pos 個位置忽略
        y_ids[t] = char_to_id['<pad>']
    return x_ids, y_ids

xy_ids = [encode_with_eos(str(s)) for s in df_train['src']]
df_train[['char_id_list', 'label_id_list']] = pd.DataFrame(xy_ids, index=df_train.index)


df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=882,882,14,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 15, 15, 9]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 15, 9, 1]"
1,(6+1)*5=35,35,10,"[2, 13, 5, 8, 3, 4, 12, 17, 10, 12]","[0, 0, 0, 0, 0, 0, 0, 10, 12, 1]"
2,13+32+29=74,74,11,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 14, 11]","[0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 1]"
3,31*(3-11)=-248,-248,14,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1]"
4,24*49+1=1181,1181,12,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 15, 8]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 15, 8, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
batch_size = 64
epochs = 2
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences) # Write your code here

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        row = self.sequences.iloc[index]
        x = row['char_id_list'] # Write your code here
        y = row['label_id_list'] # Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x, batch_first=True, padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y, batch_first=True, padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2) # Write your code here

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=char_to_id['<pad>'])

        # self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

        # self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.rnn_layer1 = torch.nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

        self.rnn_layer2 = torch.nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim), torch.nn.ReLU(), torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x, batch_x_lens, batch_first=True, enforce_sorted=False)

        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            x = torch.tensor(char_list, dtype=torch.long, device=device).unsqueeze(0)  # [T] -> [1, T]
            lens = torch.LongTensor([x.size(1)]).to(device)
            # Input the tensor to the embedding layer, LSTM layers, linear respectively
            emb = self.embedding(x)  # [1, T, E]
            packed = torch.nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
            h1, _ = self.rnn_layer1(packed)
            h2, _ = self.rnn_layer2(h1)
            out, _ = torch.nn.utils.rnn.pad_packed_sequence(h2, batch_first=True)
            logits = self.linear(out)
            y = logits[0, -1, :] # Obtain the next token prediction y

            next_char = int(torch.argmax(y).item()) # Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>']) # Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # Write your code here. Use Adam or AdamW for Optimizer

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [17]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad(set_to_none=True)

        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        # Back propagation
        loss = criterion(
            batch_pred_y.reshape(-1, batch_pred_y.size(-1)),
            batch_y.to(device).reshape(-1)
        )
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)

        # Write your code here
        optimizer.step()
        # Optimize parameters in the model

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        prediction = model.generator('1+1=') # An example of using generator: model.generator('1+1=')

        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        pred_tokens = model.generator(batch_x)
        pred_seq = ''.join(pred_tokens)
        gen_part = pred_seq[len(batch_x):]
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total
        if gen_part == str(batch_y):
            matched += 1
        total += 1

    print(matched/total)

Train epoch 1: 100%|██████████| 37020/37020 [1:38:36<00:00,  6.26it/s, loss=0.687]
Validation epoch 1: 263250it [2:03:17, 35.59it/s]


0.37886039886039885


Train epoch 2: 100%|██████████| 37020/37020 [1:45:52<00:00,  5.83it/s, loss=0.641]
Validation epoch 2: 263250it [2:05:53, 34.85it/s]

0.4232744539411206
